# SBP 1D analysis

The plots from `spider-bead-pull-ai/pipeline_overview.ipynb`, with the same plotting functions and saved 1D results. Change the run, frequency and geometry below. All processing, fitting and plotting live in `src/`.

The 1D model adjusts disk gaps and measurement-plane positions. Disk tilts and independently sloped threads are represented in the separate **3D full-tilt analysis** notebook.

Display band: **20–22 GHz**. Geometry: corrected nominal mirror–D1 gap **7.803 mm**. The imported four-plane radial-analysis results are the results used by the original notebook; they are distinct from the separate direct-complex full-tilt fit.

Saved plots are included so the notebook is useful on GitHub without measurement files. To regenerate, supply the files described in README. Optional recomputation commands are at the end.

The saved 1D fits reproduce the original notebook, whose geometry fit used the full acquired sweep. Figures select 20–22 GHz; the optional recomputation below fits only 20–22 GHz. These saved fits are not presented as a new band-limited refit.

### Editing figure text

Each figure cell explicitly lists its text arguments: titles, axis labels, legends, colorbar labels, annotations and table headings. Edit those strings and rerun that cell to refresh both the inline figure and its PNG/PDF exports. Use `""` to hide a text item. Keyword names describe what they control, such as `measured_field_label`, `disk_label_template` and `residual_y_label`.

Templates such as `{frequency_GHz:.4f}`, `{thread_index}` and `{z_mm:.3f}` retain the actual plotted values while letting you change the wording and number format. Keep those placeholders when you want the values to update automatically. LaTeX labels can use raw strings, for example `r"$|E_T|$ [V m]"`; ordinary math braces remain intact.

Six cells use `partial(...)` to create a plot function with those text settings. The interactive explorer reuses these same configured functions. After changing their text, rerun the corresponding figure cell and then the explorer cell. All rendering and data-processing code stays in `src`.


In [ ]:
from functools import partial
from pathlib import Path
from src import pipeline_plots as plots

plots.configure_style()
RUN_DIR = Path("data/spider_bead_pull_runs/run_20261025_141003")
FREQUENCY_RANGE_GHZ = (20.0, 22.0)
FREQUENCY_GHZ = 20.825
GEOMETRY = "both"  # fixed, gaps, positions, both
ZOOM_GHZ = (20.6, 21.05)
FIT_3D_PATH = None  # 3D results are shown in sbp_3d_analysis.ipynb
FULL_BEAD_FILE = Path("data/full_bead_pull/result_bf_21_0.jld2")
COMPARISON_GHZ = 21.0
PREVIOUS_1D_DIR = Path("data/outputs/run_20261025_141003/history/nominal_gap_8p803mm")
SAVE_FIGURES = False

data = plots.load_pipeline(
    RUN_DIR, fit_3d_path=FIT_3D_PATH, frequency_range_GHz=FREQUENCY_RANGE_GHZ,
)
FIGURE_DIR = data.output_dir / "plots" if SAVE_FIGURES else None

In [ ]:
plots.summary(data)

## 1. Physical setup

The longitudinal coordinate starts at the mirror and increases towards the antenna. The three gaps are surface to surface; each disk is 1 mm thick. T0–T3 label the four measured central diameters. The antenna distance is annotated separately so the millimetre-scale stack remains readable.

The corrected surface gaps are **7.803, 8.853, 8.855 mm**. This view reads the current Stage 1 geometry, independently of the saved fit files.

In [ ]:
plots.present(
    plots.plot_setup(
        data,
        title="Corrected measurement geometry · mirror–disk 1 gap 7.803 mm",
        x_label="z from mirror [mm]",
        mirror_label="Mirror",
        disk_label_template="Disk {disk_index}",
        thread_label_template="T{thread_index}",
        thread_position_template="{z_mm:.3f}",
        gap_width_template="{gap_mm:.3f} mm",
        antenna_label_template="Antenna: {antenna_distance_m:.3f} m from mirror  →",
        antenna_unknown_label="Antenna lies towards +z  →",
        table_headers=("Plane", "z [mm]", "Line [cm]", "R int. [cm]"),
        table_thread_template="T{thread_index}",
        table_position_template="{z_mm:.3f}",
        table_length_template="{length_cm:.2f}",
        table_radius_template="{radius_cm:.2f}",
        missing_value_label="—",
        properties_template="{disk_count} disks · {disk_thickness_mm:g} mm thick · εᵣ = {disk_eps_r:g}\nDisk radius {disk_radius_cm:g} cm\nBead Ø {bead_diameter_mm:g} mm · εᵣ = {bead_eps_r:g}\nSaved input-power normalization: {input_power_W:g} W",
        footnote="Surface-to-surface gaps; +z points towards the antenna. Antenna offset is not drawn to scale. R int. is the radial integration limit.",
    ),
    '01_setup', FIGURE_DIR,
)

## 2. Raw measurement across the sweep

Each panel shows the acquired reflection magnitude along one calibrated line. Grey end regions are the 2.5 cm off-beam references. All panels share the same colour scale. The parameter label is read from the acquisition metadata: the default run measured **S22**, despite the historical HDF5 filename.

In [ ]:
plots.present(
    plots.plot_measurement_map(
        data,
        title="Raw measurement · {s_parameter}",
        panel_title_template="{thread_name} · {z_mm:.3f} mm from mirror",
        x_label="Frequency [GHz]",
        y_label="Distance from calibrated start [cm]",
        colorbar_label="{s_parameter} magnitude [dB]",
    ),
    '02_raw_measurement_map', FIGURE_DIR,
)

## 3. Complex raw traces at the selected frequency

Real and imaginary components retain the phase information needed for field reconstruction. The horizontal coordinate is distance along the calibrated line, before recentering.

In [ ]:
plots.present(
    plots.plot_raw_traces(
        data, FREQUENCY_GHZ,
        title="Raw traces · {frequency_GHz:.4f} GHz · shaded off-beam references",
        panel_title_template="{thread_name} · {z_mm:.3f} mm from mirror",
        x_label="Distance from calibrated start [cm]",
        y_label="{s_parameter} [dimensionless]",
        real_label="Real",
        imaginary_label="Imaginary",
    ),
    '03_raw_traces', FIGURE_DIR,
)

## 4. Off-beam reference and subtraction

A complex affine baseline is fitted to both 2.5 cm edge regions. These panels show that fitted baseline against the raw data. The analysis uses $\Delta\Gamma=\Gamma-\Gamma_\mathrm{ref}$ in the interior and sets the reference regions to zero.

In [ ]:
reference_subtraction_plot = partial(
    plots.plot_reference_subtraction,
    title="Off-beam reference subtraction · {frequency_GHz:.4f} GHz",
    panel_title_template="{thread_name} · {z_mm:.3f} mm from mirror · {component}",
    x_label="Distance from calibrated start [cm]",
    real_y_label="Re {s_parameter}",
    imaginary_y_label="Im {s_parameter}",
    real_component_label="Real",
    imaginary_component_label="Imaginary",
    measured_label="Raw {s_parameter}",
    baseline_label="Complex affine baseline",
)
plots.present(reference_subtraction_plot(data, FREQUENCY_GHZ), '04_reference_subtraction', FIGURE_DIR)

## 5. Beam centre and two-half symmetry

The saved beam centre was estimated jointly from the frequency sweep for each line. Folding the left and right halves around that centre shows how well the radial-symmetry assumption describes the complex perturbation. These are the two halves before averaging; no new centre fit is performed here.

In [ ]:
beam_center_plot = partial(
    plots.plot_beam_center,
    title="Beam-centre symmetry · {frequency_GHz:.4f} GHz",
    panel_title_template="{thread_name} · {z_mm:.3f} mm from mirror · centre {center_cm:.2f} cm",
    x_label="Distance from saved centre [cm]",
    y_label="ΔΓ component [dimensionless]",
    left_real_label="Left real",
    left_imaginary_label="Left imaginary",
    right_real_label="Right real",
    right_imaginary_label="Right imaginary",
)
plots.present(beam_center_plot(data, FREQUENCY_GHZ), '05_beam_centre', FIGURE_DIR)

## 6. From the folded perturbation to a radial field

The analysis averages the complex perturbations at equal radii, unwraps their phase and takes the calibrated complex square root. The resulting effective field still contains the finite-bead response. The hollow endpoint is forced to zero for integration. Phase is hidden where the field is very weak; each line retains an unresolved overall sign, so phases from different lines are not an independently measured longitudinal phase.

In [ ]:
radial_fields_plot = partial(
    plots.plot_radial_fields,
    title="Finite-bead effective radial fields · {frequency_GHz:.4f} GHz",
    amplitude_title_template="{thread_name} · {z_mm:.3f} mm from mirror",
    phase_title="Saved radial square-root branch",
    x_label="Radius [cm]",
    amplitude_y_label="Effective |E| [V/m]",
    phase_y_label="Effective field phase [rad]",
    endpoint_label="Forced-zero integration endpoint",
)
plots.present(radial_fields_plot(data, FREQUENCY_GHZ), '06_radial_fields', FIGURE_DIR)

## 7. Transverse field maps under radial symmetry

These maps revolve each measured radial profile into $E(x,y)=E(\sqrt{x^2+y^2})$. They are inferred maps from one central diameter per plane. The shared colour scale allows comparison between planes; areas outside each integration radius are blank.

In [ ]:
transverse_maps_plot = partial(
    plots.plot_transverse_maps,
    title="Radial-symmetry reconstruction · {frequency_GHz:.4f} GHz\nOne central diameter measured per plane",
    panel_title_template="{thread_name} · {z_mm:.3f} mm from mirror",
    x_label="Recentered x [cm]",
    y_label="Recentered y [cm]",
    colorbar_label="Effective |E| [V/m]",
)
plots.present(transverse_maps_plot(data, FREQUENCY_GHZ), '07_transverse_maps', FIGURE_DIR)

## 8. Area weighting and cumulative transverse integration

The transverse integral is $E_T(z,f)=2\pi\int_0^R rE(r,z,f)\,dr$. The left panels show the complex area-weighted integrand; the right panels accumulate it outwards. Integrating the complex field preserves radial phase cancellation. Radius labels are in centimetres, while the calculation uses metres.

In [ ]:
plots.present(
    plots.plot_radial_integration(
        data, FREQUENCY_GHZ,
        title="Transverse integration · {frequency_GHz:.4f} GHz",
        integrand_title_template="{thread_name} · {z_mm:.3f} mm from mirror",
        cumulative_title="Cumulative complex transverse integral",
        x_label="Radius [cm]",
        integrand_y_label="2πr E(r) [V]",
        cumulative_y_label="Integral [V m]",
        real_label="Real",
        imaginary_label="Imaginary",
        magnitude_label="Magnitude",
    ),
    '08_radial_integration', FIGURE_DIR,
)

## 9. Integrated transverse fields across frequency

These are the saved effective $E_T$ values at the four measurement planes. Their units are **V m**, whereas local radial fields have units **V/m**. The 1D fit uses their magnitudes; the phase panel displays the saved square-root branches.

In [ ]:
plots.present(
    plots.plot_integrated_fields(
        data,
        title="Transversely integrated effective field",
        phase_title="",
        x_label="Frequency [GHz]",
        amplitude_y_label="|ET| [V m]",
        phase_y_label="Saved branch phase [rad]",
        thread_label_template="{thread_name} · {z_mm:.3f} mm from mirror",
    ),
    '09_integrated_fields', FIGURE_DIR,
)

## 10. How much radial phase cancellation matters

The ratio $|\int E\,dA|/\int |E|\,dA$ approaches one for a phase-aligned transverse profile and decreases when different radii cancel. This helps explain why a large local field need not produce a large integrated field.

In [ ]:
plots.present(
    plots.plot_transverse_coherence(
        data,
        title=r"Transverse phase coherence: $|\int E\,dA| / \int |E|\,dA$",
        x_label="Frequency [GHz]",
        y_label="Coherent / amplitude-only integral",
        thread_label_template="{thread_name} · {z_mm:.3f} mm from mirror",
    ),
    '10_transverse_coherence', FIGURE_DIR,
)

## 11. Representative 1D fit

The orange markers are measured transverse-field magnitudes at the fitted plane positions. Magenta is the finite-bead prediction fitted to those magnitudes; green is the model with the bead response removed. Grey bands show fitted disks, dotted outlines retain the corrected nominal disks, and hollow markers retain the corrected nominal plane positions. The finite-bead curve is drawn only where the whole bead fits in vacuum. The lower panel shows signed residuals normalized by the four-plane data norm.

This fit uses the corrected nominal geometry. Its markers and model use the fitted coordinates; hollow markers retain the corrected starting plane positions.

In [ ]:
diagnostic_plot = partial(
    plots.plot_1d_diagnostic,
    title_template="{geometry_label} · {frequency_label} · refit with corrected geometry",
    x_label="z from mirror [mm]",
    y_label="$|E_T|$ [V m]",
    residual_y_label="Residual /\n‖data‖ [%]",
    true_field_label="Deconvolved field",
    bead_model_label="Finite-bead model",
    nominal_position_label="Corrected nominal z",
    measured_field_label="Measured |ET|",
    disk_label_template="D{disk_index}",
    thread_label_template="T{thread_index}",
    residual_annotation_template="Relative residual {residual_percent:.1f}%",
    geometry_labels={"fixed": "Fixed geometry", "gaps": "Fit gaps", "positions": "Fit plane positions", "both": "Fit gaps + planes"},
    frequency_label_template="{frequency_GHz:.6g} GHz",
    nearest_frequency_label_template="{frequency_GHz:.6g} GHz (nearest to {requested_frequency_GHz:.6g} GHz)",
    footnote="Residual = finite-bead model − measured magnitude. Bead curves require the whole bead in vacuum; dotted disk outlines mark the corrected nominal geometry (first gap 7.803 mm).",
)
plots.present(diagnostic_plot(data, FREQUENCY_GHZ, GEOMETRY), '11_1d_diagnostic', FIGURE_DIR)

## 12. Compare the four geometry choices

Each panel displays its own saved fit: fully fixed geometry, adjustable gaps, adjustable measurement planes, or both adjustable. Geometry is shared across frequencies; amplitudes are fitted at each frequency. No fit is rerun when switching views.

In [ ]:
plots.present(
    plots.plot_1d_comparison(
        data, FREQUENCY_GHZ,
        title="Refitted 1D fields · corrected nominal geometry",
        panel_title_template="{geometry_label} · {frequency_label}",
        x_label="z from mirror [mm]",
        y_label="$|E_T|$ [V m]",
        true_field_label="Deconvolved field",
        bead_model_label="Finite-bead model",
        nominal_position_label="Corrected nominal z",
        measured_field_label="Measured |ET|",
        thread_label_template="T{thread_index}",
        residual_annotation_template="Relative residual {residual_percent:.1f}%",
        geometry_labels={"fixed": "Fixed geometry", "gaps": "Fit gaps", "positions": "Fit plane positions", "both": "Fit gaps + planes"},
        frequency_label_template="{frequency_GHz:.6g} GHz",
        nearest_frequency_label_template="{frequency_GHz:.6g} GHz (nearest to {requested_frequency_GHz:.6g} GHz)",
        missing_fit_template="{geometry_label}\nNo saved result",
        footnote="Each panel uses its own saved geometry and amplitude. Filled measurements use fitted z; hollow points and dotted disk outlines retain the corrected nominal locations.",
    ),
    '12_1d_comparison', FIGURE_DIR,
)

## 13. Fitted longitudinal field in the selected band

The model is evaluated on a dense longitudinal grid using the saved amplitude and geometry. The colour represents the magnitude of the deconvolved transverse integral on a logarithmic scale. Disk boundaries and measurement planes locate the sampled points within the standing-wave pattern.

In [ ]:
plots.present(
    plots.plot_1d_field_map(
        data, GEOMETRY,
        logarithmic=True,
        title_template="Refitted field · {geometry_label} · corrected geometry",
        x_label="Frequency [GHz]",
        y_label="z from mirror [mm]",
        colorbar_label="$|E_T(z,f)|$ [V m]",
        logarithmic_colorbar_suffix=" · logarithmic color scale",
        thread_label_template="T{thread_index}",
        geometry_labels={"fixed": "Fixed geometry", "gaps": "Fit gaps", "positions": "Fit plane positions", "both": "Fit gaps + planes"},
        footnote="White bands outline fitted disks; dotted outlines/lines retain the corrected nominal geometry. Colored dashed lines locate fitted measurement planes.",
    ),
    '13_1d_field_map', FIGURE_DIR,
)

## 14. Longitudinal field near the selected resonance band

A linear colour scale and the configured zoom band make the resonance structure easier to inspect. This is the same saved fit as the preceding map.

In [ ]:
plots.present(
    plots.plot_1d_field_map(
        data, GEOMETRY,
        frequency_range_GHz=ZOOM_GHZ,
        title_template="Refitted field · {geometry_label} · corrected geometry",
        x_label="Frequency [GHz]",
        y_label="z from mirror [mm]",
        colorbar_label="$|E_T(z,f)|$ [V m]",
        logarithmic_colorbar_suffix=" · logarithmic color scale",
        thread_label_template="T{thread_index}",
        geometry_labels={"fixed": "Fixed geometry", "gaps": "Fit gaps", "positions": "Fit plane positions", "both": "Fit gaps + planes"},
        footnote="White bands outline fitted disks; dotted outlines/lines retain the corrected nominal geometry. Colored dashed lines locate fitted measurement planes.",
    ),
    '14_1d_field_map_zoom', FIGURE_DIR,
)

## 15. Geometry shifts from the corrected nominal setup

These are the refitted changes relative to the corrected geometry (**nominal first gap 7.803 mm**). Dashed lines are the ±0.5 mm optimization bounds, not uncertainty intervals. The archived fits retain their old nominal geometry and are compared separately below.

In [ ]:
plots.present(
    plots.plot_geometry_shifts(
        data,
        title="Refitted shifts from corrected geometry (first gap 7.803 mm)",
        gap_panel_title="Surface-to-surface gap shifts",
        position_panel_title="Measurement-plane shifts",
        y_label="Shift from nominal [µm]",
        gap_tick_template="Gap {gap_index}",
        thread_tick_template="T{thread_index}",
        value_annotation_template="{shift_um:.0f}",
        zero_shift_label="0",
        geometry_labels={"fixed": "Fixed geometry", "gaps": "Fit gaps", "positions": "Fit plane positions", "both": "Fit gaps + planes"},
        bound_label="Allowed shift bounds",
        footnote="The dashed limits constrain optimization; they are not uncertainties. Positive shifts increase a gap or move a plane away from the mirror.",
    ),
    '15_geometry_shifts', FIGURE_DIR,
)

## 16. 1D boost spectra and fit quality

The upper panel shows reconstructed power boost $\beta^2$ for the four geometry assumptions; the lower panel shows $\|\mathrm{model}-|E_T|\|_2/\||E_T|\|_2$ across planes at each frequency. These spectra inherit the model and input-power assumptions used by the saved fits.

In [ ]:
plots.present(
    plots.plot_fit_spectra(
        data,
        title="Refitted 1D boost and fit quality · corrected geometry",
        x_label="Frequency [GHz]",
        boost_y_label=r"Reconstructed boost power $\beta^2$",
        residual_y_label="Relative residual [%]",
        geometry_labels={"fixed": "Fixed geometry", "gaps": "Fit gaps", "positions": "Fit plane positions", "both": "Fit gaps + planes"},
        footnote="Each fit has one geometry shared across the spectrum. Relative residual = ‖model − |ET|‖₂ / ‖|ET|‖₂ across measurement planes.",
    ),
    '16_1d_boost_spectra', FIGURE_DIR,
)

## 17. Boost and residuals in the resonance band

The configured zoom band resolves differences between the four reconstructed resonances. The residual panel provides the fit-quality context for those differences.

In [ ]:
plots.present(
    plots.plot_fit_spectra(
        data,
        frequency_range_GHz=ZOOM_GHZ,
        title="Refitted 1D boost and fit quality · corrected geometry",
        x_label="Frequency [GHz]",
        boost_y_label=r"Reconstructed boost power $\beta^2$",
        residual_y_label="Relative residual [%]",
        geometry_labels={"fixed": "Fixed geometry", "gaps": "Fit gaps", "positions": "Fit plane positions", "both": "Fit gaps + planes"},
        footnote="Each fit has one geometry shared across the spectrum. Relative residual = ‖model − |ET|‖₂ / ‖|ET|‖₂ across measurement planes.",
    ),
    '17_1d_boost_zoom', FIGURE_DIR,
)

## 17a. Effect of the corrected nominal geometry

Each row compares one geometry option: **solid** curves use the new 7.803 mm nominal mirror gap and corrected planes; **dashed** curves use the archived 8.803 mm nominal geometry. The measured transverse fields, fitting band (12–22 GHz), bounds (±0.5 mm) and penalty (0.001) are the same. Only 20–22 GHz is displayed.

The right panels show the relative magnitude residual at each frequency, not an uncertainty. Numerical summaries for the full fitting band, the displayed band and 21 GHz are saved in `fit_1D_refit_comparison.json` and `.csv` beside the current results. Several fitted geometry parameters still reach their bounds; lower residuals do not establish a unique physical geometry.

In [ ]:
plots.present(
    plots.plot_1d_refit_comparison(
        data, PREVIOUS_1D_DIR,
        title="1D refits · corrected versus previous nominal geometry",
        panel_title_template="{geometry_label} · {quantity_label}",
        x_label="Frequency [GHz]",
        boost_y_label="Power boost β²",
        residual_y_label="Relative magnitude residual [%]",
        current_label="Refit · {current_gap_mm:.3f} mm nominal mirror gap",
        previous_label="Previous · {previous_gap_mm:.3f} mm nominal mirror gap",
        boost_panel_label="Power boost",
        residual_panel_label="Magnitude residual",
        geometry_labels={"fixed": "Fixed geometry", "gaps": "Fit gaps", "positions": "Fit plane positions", "both": "Fit gaps + planes"},
    ),
    '30_1d_refit_comparison', FIGURE_DIR,
)

## 18. Which planes contribute to the 1D residuals?

Positive colours mean that the model exceeds the measured magnitude; negative colours mean it falls below it. All four cases use a shared colour scale. Each frequency column is normalized by its four-plane data norm, avoiding division by an individual plane near a node.

In [ ]:
plots.present(
    plots.plot_1d_residual_map(
        data,
        title="Refitted 1D residuals · corrected geometry",
        x_label="Frequency [GHz]",
        y_label="Sub-thread",
        colorbar_label="(Model − measured magnitude) / measured norm [%]",
        fixed_panel_title="Fixed",
        gaps_panel_title="Gaps",
        positions_panel_title="Positions",
        both_panel_title="Both",
        thread_tick_template="{thread_index}",
        missing_title="No saved 1D fits",
        missing_message="Run src.fit_1D on integrated_fields.npz first.",
    ),
    '18_1d_residual_maps', FIGURE_DIR,
)

## 19. Independent measured beam reference

The copied S31 reference supplies 1/e field-amplitude radii and their mean over six scan planes. These widths provide an initial guess for the 3D modes within the measured reference band. Its scan coordinates are not registered to this booster, and its uncalibrated amplitudes, centres and missing wavefront phase are not imposed on the fit.

In [ ]:
plots.present(
    plots.plot_beam_reference(
        data, FREQUENCY_GHZ,
        title="Beam-width initialization · reference z is not registered to the booster",
        spectrum_title="Measured S31 beam-width fits",
        planes_title="Reference scan planes · {frequency_GHz:.3f} GHz",
        frequency_x_label="Reference frequency [GHz]",
        position_x_label="Reference scan z [cm]",
        y_label="1/e field-amplitude radius [cm]",
        reference_label_template="Reference z={z_cm:.0f} cm",
        mean_label="Mean used for initial guess",
    ),
    '19_beam_reference', FIGURE_DIR,
)

## 22. Full bead-pull comparison

The supplied `result_bf_21_0.jld2` contains a **20.5–21.5 GHz sweep**, with 28 longitudinal samples of the transverse integral and 100 saved Monte Carlo fit draws per frequency. The reader decodes the numeric JLD2 results into a small cache under `data/outputs/full_bead_pull/`. The original result remains unchanged.

The figures compare the full scan with all four spider 1D boost results and the selected spider geometry for field profiles. The full-tilt 3D analysis is displayed separately in sbp_3d_analysis.ipynb. This full-pull archive contains $E_T=\int E\,dA$ in **V m**, not local transverse field maps in V/m.

Coordinates are aligned at each setup's mirror: full-pull $u=z_m(f)-z_\mathrm{saved}$ versus spider's saved outward coordinate. The two measurements have different geometries and z sampling. Curves reconstructed from the full scan's mean parameters retain the individual thickness, permittivity and loss of each disk. They are **not** Monte Carlo mean field curves; the boost curve is read directly from the saved Monte Carlo result.

**Absolute values retain their saved normalization.** The archive's boost normalization differs from the spider formula, and its physical power/correction metadata are incomplete. The audit below records the numerical difference; no rescaling has been assumed. Separate normalized panels compare shapes without claiming equal input power. No cross-run complex phase alignment is imposed.

In [ ]:
from src import full_bead_comparison as comparison

full_bead = comparison.load_full_bead(FULL_BEAD_FILE)
COMPARISON_DIR = FIGURE_DIR / "full_bead_comparison" if FIGURE_DIR is not None else None

In [ ]:
comparison.normalization_summary(full_bead)
comparison.comparison_summary(data, full_bead, COMPARISON_GHZ)

### Boost spectra and resonance shapes

The upper panel compares saved power boosts. The lower panel independently normalizes each 1D spectrum to its maximum in the comparison band, making resonance frequencies and widths easier to compare despite normalization differences. The full-tilt 3D results are shown in their own notebook.

In [ ]:
plots.present(
    comparison.plot_boost_comparison(
        data, full_bead,
        title="Boost comparison · spider 1D refits with corrected geometry",
        shape_title="Spectral shape and resonance position · each 1D curve normalized separately",
        x_label="Frequency [GHz]",
        y_label="Saved power boost β²",
        shape_y_label="β² / own maximum in comparison band",
        full_curve_label="Full bead pull · as saved",
        full_shape_label="Full bead pull",
        spider_curve_label_template="Spider 1D · {mode}",
        trial_curve_label="Spider 3D trial · receive-coupled",
    ),
    '22_boost_comparison', COMPARISON_DIR,
)

### Field profiles at 21 GHz

The left panels show the measured effective transverse integrals, finite-bead predictions and deconvolved model fields in their saved units. On the right, effective-field curves and points are divided by each scan's maximum measured magnitude; deconvolved curves are divided by their own maximum over the displayed z range. Filled and outlined disk regions distinguish the two fitted geometries. The selected frequency follows COMPARISON_GHZ and is snapped to a common saved bin.

In [ ]:
plots.present(
    comparison.plot_field_comparison(
        data, full_bead, COMPARISON_GHZ, GEOMETRY,
        title="Full versus spider bead pull · common saved frequency {frequency_GHz:.4f} GHz\nGrey disks: full fit · dotted: spider refit with corrected geometry",
        x_label="Distance outward from each fitted mirror [mm]",
        y_label="|ET| [V m] · as saved",
        normalized_effective_y_label="|ET| / own maximum measured |ET|",
        normalized_field_y_label="|ET| / own maximum over shown z",
        effective_field_title="Finite-bead effective field",
        normalized_effective_field_title="Effective field · shape comparison",
        deconvolved_field_title="Deconvolved field",
        normalized_deconvolved_field_title="Deconvolved field · shape comparison",
        full_bead_label="Full · bead model at mean parameters",
        spider_bead_label_template="Spider {geometry} · bead model",
        full_measured_label_template="Full · {n_full} measured integrals",
        spider_measured_label_template="Spider · {n_spider} measured integrals",
        full_field_label="Full · model at mean parameters",
        spider_field_label_template="Spider {geometry} · model",
    ),
    '23_field_comparison_21ghz', COMPARISON_DIR,
)

### Field profiles near the full-pull resonance

This second profile comparison uses the largest saved full-pull boost on the **common acquired frequency grid**. The full archive peaks at 20.816 GHz; the common grid selects 20.820 GHz. Both field models and both sets of measured integrals use exactly the same frequency shown in the figure.

In [ ]:
plots.present(
    comparison.plot_field_comparison(
        data, full_bead, comparison.reference_peak_frequency(data, full_bead), GEOMETRY,
        title="Full versus spider bead pull · common saved frequency {frequency_GHz:.4f} GHz\nGrey disks: full fit · dotted: spider refit with corrected geometry",
        x_label="Distance outward from each fitted mirror [mm]",
        y_label="|ET| [V m] · as saved",
        normalized_effective_y_label="|ET| / own maximum measured |ET|",
        normalized_field_y_label="|ET| / own maximum over shown z",
        effective_field_title="Finite-bead effective field",
        normalized_effective_field_title="Effective field · shape comparison",
        deconvolved_field_title="Deconvolved field",
        normalized_deconvolved_field_title="Deconvolved field · shape comparison",
        full_bead_label="Full · bead model at mean parameters",
        spider_bead_label_template="Spider {geometry} · bead model",
        full_measured_label_template="Full · {n_full} measured integrals",
        spider_measured_label_template="Spider · {n_spider} measured integrals",
        full_field_label="Full · model at mean parameters",
        spider_field_label_template="Spider {geometry} · model",
    ),
    '24_field_comparison_resonance', COMPARISON_DIR,
)

### Geometry and longitudinal sampling

The full measurement has four groups of seven nearby z positions; the spider measurement has one central diameter at each of four z positions. Each setup uses its own fitted disk locations. Alignment at the mirror does not imply identical geometries or coincident samples.

In [ ]:
plots.present(
    comparison.plot_geometry_comparison(
        data, full_bead, COMPARISON_GHZ, GEOMETRY,
        title="Geometry and sampling · {frequency_GHz:.4f} GHz\nFull absolute mirror coordinate {full_mirror_z_m:.6f} m; spider nominal mirror coordinate {spider_mirror_z_m:.3f} m",
        x_label="Distance outward from each fitted mirror [mm]",
        full_panel_title_template="Full bead pull · fitted geometry · {n_positions} sampled z positions",
        spider_panel_title_template="Spider · {geometry} refit with corrected geometry · {n_positions} sampled z positions",
        disk_label_template="D{disk_index}",
        thread_label_template="T{thread_index}",
    ),
    '25_geometry_comparison', COMPARISON_DIR,
)

### Fitted gap spectra

The full-pull archive fits geometry independently at each frequency. Each spider case has one geometry shared across the sweep. These curves compare the resulting surface-to-surface gaps; they are not uncertainty bands.

In [ ]:
plots.present(
    comparison.plot_gap_comparison(
        data, full_bead,
        title="Fitted gaps · spider refits with corrected geometry",
        x_label="Frequency [GHz]",
        y_label_template="Gap {gap_index} [mm]",
        full_curve_label="Full · fitted per frequency",
        spider_curve_label_template="Spider · {mode}",
    ),
    '26_gap_comparison', COMPARISON_DIR,
)

### Deconvolved field maps in saved units

Both model fields are evaluated on the same longitudinal grid at the 201 common acquired frequencies. The shared colour scale retains the absolute stored magnitudes. The full model uses its frequency-dependent fitted gaps and individual disk properties; the spider model uses its shared fitted geometry.

In [ ]:
plots.present(
    comparison.plot_field_map_comparison(
        data, full_bead, GEOMETRY,
        title="Field magnitudes · spider refit with corrected geometry",
        normalized_title="Longitudinal field patterns · independently normalized columns",
        x_label="Common acquired frequency [GHz]",
        y_label="Distance outward from mirror [mm]",
        colorbar_label="Deconvolved |ET| [V m] · as saved",
        normalized_colorbar_label="|ET| / own maximum at each frequency",
        full_panel_title="Full · model at mean parameters",
        spider_panel_title_template="Spider · {geometry}",
    ),
    '27_field_map_comparison', COMPARISON_DIR,
)

### Deconvolved field shapes

Each frequency column is divided by its own maximum field over the displayed z range. This exposes standing-wave nodes and relative field distributions while removing the overall amplitude scale. These are model reconstructions, not additional measured z samples.

In [ ]:
plots.present(
    comparison.plot_field_map_comparison(
        data, full_bead, GEOMETRY,
        normalized=True,
        title="Longitudinal field magnitudes · shared colour scale; saved normalizations",
        normalized_title="Field shapes · spider refit with corrected geometry",
        x_label="Common acquired frequency [GHz]",
        y_label="Distance outward from mirror [mm]",
        colorbar_label="Deconvolved |ET| [V m] · as saved",
        normalized_colorbar_label="|ET| / own maximum at each frequency",
        full_panel_title="Full · model at mean parameters",
        spider_panel_title_template="Spider · {geometry}",
    ),
    '28_field_map_shapes', COMPARISON_DIR,
)

### Fit quality alongside the comparison

The top panel uses the same dimensionless magnitude-residual definition for both scans, $\|\mathrm{model}-|E_T|\|_2/\||E_T|\|_2$, evaluated on each scan's own measured z points. For the full pull this is the residual of the mean-parameter model.

In [ ]:
plots.present(
    comparison.plot_fit_quality_comparison(
        data, full_bead,
        title="Magnitude residuals · spider refits with corrected geometry",
        trial_title="3D trial uses a different observable · no matching full-pull spatial data",
        x_label="Frequency [GHz]",
        y_label="Magnitude residual [%]",
        trial_y_label="Weighted complex-Q residual [%]",
        full_curve_label_template="Full · {n_full} planes, mean-parameter model",
        spider_curve_label_template="Spider {mode} · {n_spider} planes",
        trial_curve_label="Spider 3D trial · saved frequencies",
    ),
    '29_fit_quality_comparison', COMPARISON_DIR,
)

## Interactive explorer

With a running kernel and widget support, choose a view, frequency and 1D geometry case below. The slider updates after release; 3D views select the nearest saved 3D frequency. The static figures above remain visible without widgets. To export a different frequency, change the configuration cell and rerun the desired plot cells.

In [ ]:
SHOW_INTERACTIVE = False
if SHOW_INTERACTIVE:
    plots.interactive_explorer(
        data, FREQUENCY_GHZ, GEOMETRY,
        plot_functions={
            "diagnostic": diagnostic_plot,
            "radial": radial_fields_plot,
            "reference": reference_subtraction_plot,
            "center": beam_center_plot,
            "transverse": transverse_maps_plot,
        },
        frequency_control_label="GHz",
        geometry_control_label="1D case",
        view_control_label="View",
        diagnostic_view_label="1D field + setup",
        radial_view_label="Radial field",
        reference_view_label="Reference subtraction",
        center_view_label="Beam-centre symmetry",
        transverse_view_label="Transverse reconstruction",
        spatial_view_label="3D saved profiles",
        fixed_geometry_label="fixed",
        gaps_geometry_label="gaps",
        positions_geometry_label="positions",
        both_geometry_label="both",
    )


## Recompute the analysis (optional)

Plot cells above read existing results; they do not refit each time. The following functions reproduce the original raw-line reconstruction and 1D fits. Enable them when analysing a new run. Geometry options are `fixed`, `gaps`, `positions`, `both`; configuration and fitting bounds are those of the original Python model.

In [ ]:
from src.sbp_analysis import analyze_run
from src.fit_1D import fit_all_geometries
import numpy as np

RUN_ANALYSIS = False
if RUN_ANALYSIS:
    fields_file = analyze_run(RUN_DIR)
    with np.load(fields_file) as measured:
        keep = (measured["frequencies_Hz"] >= 20e9) & (measured["frequencies_Hz"] <= 22e9)
        fits = fit_all_geometries(
            measured["frequencies_Hz"][keep], measured["z_from_mirror_m"], measured["ET"][:, keep],
        )
    for geometry, result in fits.items():
        result.save(fields_file.with_name(f"integrated_fields_fit_1D_{geometry}.npz"))